# Common Job Identity Utilities

**Purpose:** Helpers for job identification, matching, and lifecycle management.

**Usage:**
```python
%run "./common/common_job_identity"

# Get execution context
context = get_execution_context()
print(f"Running in: {context['notebook_path']}")
print(f"Job ID: {context['job_id']}")
print(f"Run ID: {context['run_id']}")

# Generate unique run identifier
run_id = generate_run_id(prefix="ingestion")
```

**Features:**
* Execution context detection
* Job/run ID extraction
* Unique identifier generation
* Notebook/job context helpers

In [0]:
import uuid
import os
from datetime import datetime
from typing import Optional, Dict, Any

# ============================================================================
# JOB IDENTITY UTILITIES
# ============================================================================

In [0]:
def get_execution_context() -> Dict[str, Any]:
    """
    Get current execution context (job, notebook, user, etc.).
    
    Returns:
        Dictionary with execution context information
    
    Example:
        context = get_execution_context()
        print(f"Notebook: {context['notebook_path']}")
        print(f"User: {context['user']}")
        print(f"Is Job: {context['is_job']}")
    """
    context = {
        "notebook_path": None,
        "notebook_name": None,
        "user": None,
        "job_id": None,
        "run_id": None,
        "parent_run_id": None,
        "is_job": False,
        "cluster_id": None,
        "workspace_url": None
    }
    
    try:
        # Get notebook path
        notebook_path = dbutils.notebook.entry_point.getDbutils() \
            .notebook().getContext().notebookPath().get()
        context["notebook_path"] = notebook_path
        context["notebook_name"] = notebook_path.split("/")[-1] if notebook_path else None
    except:
        pass
    
    try:
        # Get user
        user = dbutils.notebook.entry_point.getDbutils() \
            .notebook().getContext().userName().get()
        context["user"] = user
    except:
        pass
    
    try:
        # Get job information
        tags = dbutils.notebook.entry_point.getDbutils() \
            .notebook().getContext().tags().get()
        
        # Check if running in a job
        context["is_job"] = "jobId" in tags
        
        if context["is_job"]:
            context["job_id"] = tags.get("jobId")
            context["run_id"] = tags.get("runId")
            context["parent_run_id"] = tags.get("parentRunId")
        
        # Get cluster ID
        context["cluster_id"] = tags.get("clusterId")
    except:
        pass
    
    try:
        # Get workspace URL
        api_url = dbutils.notebook.entry_point.getDbutils() \
            .notebook().getContext().apiUrl().get()
        context["workspace_url"] = api_url
    except:
        pass
    
    return context

def is_running_in_job() -> bool:
    """
    Check if notebook is running as part of a Databricks Job.
    
    Returns:
        True if running in a job, False otherwise
    
    Example:
        if is_running_in_job():
            print("Running in scheduled job")
        else:
            print("Running interactively")
    """
    context = get_execution_context()
    return context["is_job"]

def get_job_id() -> Optional[str]:
    """
    Get current job ID if running in a job.
    
    Returns:
        Job ID or None
    
    Example:
        job_id = get_job_id()
        if job_id:
            print(f"Job ID: {job_id}")
    """
    context = get_execution_context()
    return context["job_id"]

def get_run_id() -> Optional[str]:
    """
    Get current run ID if running in a job.
    
    Returns:
        Run ID or None
    
    Example:
        run_id = get_run_id()
        if run_id:
            print(f"Run ID: {run_id}")
    """
    context = get_execution_context()
    return context["run_id"]

def get_notebook_name() -> Optional[str]:
    """
    Get current notebook name.
    
    Returns:
        Notebook name or None
    
    Example:
        name = get_notebook_name()
        print(f"Running notebook: {name}")
    """
    context = get_execution_context()
    return context["notebook_name"]

In [0]:
def generate_run_id(
    prefix: Optional[str] = None,
    include_timestamp: bool = True
) -> str:
    """
    Generate a unique run identifier.
    
    Args:
        prefix: Optional prefix for the ID
        include_timestamp: Include timestamp in ID
    
    Returns:
        Unique run identifier
    
    Example:
        run_id = generate_run_id(prefix="bronze_ingestion")
        # Returns: "bronze_ingestion_20240101_123045_a1b2c3d4"
    """
    unique_part = str(uuid.uuid4()).split("-")[0]
    
    parts = []
    
    if prefix:
        parts.append(prefix)
    
    if include_timestamp:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        parts.append(timestamp)
    
    parts.append(unique_part)
    
    return "_".join(parts)

def generate_batch_id(
    layer: str = "bronze",
    dataset: Optional[str] = None
) -> str:
    """
    Generate a batch identifier for pipeline processing.
    
    Args:
        layer: Pipeline layer (bronze, silver, gold)
        dataset: Optional dataset name
    
    Returns:
        Batch identifier
    
    Example:
        batch_id = generate_batch_id(layer="bronze", dataset="orders")
        # Returns: "bronze_orders_20240101_123045_a1b2c3d4"
    """
    parts = [layer]
    
    if dataset:
        parts.append(dataset)
    
    prefix = "_".join(parts)
    return generate_run_id(prefix=prefix)

def get_or_create_pipeline_run_id() -> str:
    """
    Get existing job/run ID or create new unique ID.
    
    Returns:
        Pipeline run identifier
    
    Example:
        run_id = get_or_create_pipeline_run_id()
        # If in job: returns job run ID
        # If interactive: generates new UUID
    """
    run_id = get_run_id()
    
    if run_id:
        return str(run_id)
    else:
        return str(uuid.uuid4())

In [0]:
def print_execution_context() -> None:
    """
    Print execution context in readable format.
    
    Useful for debugging and logging.
    
    Example:
        print_execution_context()
    """
    context = get_execution_context()
    
    print("="*80)
    print("EXECUTION CONTEXT".center(80))
    print("="*80)
    
    print(f"\nNotebook: {context['notebook_path']}")
    print(f"Name: {context['notebook_name']}")
    print(f"User: {context['user']}")
    print(f"\nExecution Mode: {'Job' if context['is_job'] else 'Interactive'}")
    
    if context["is_job"]:
        print(f"Job ID: {context['job_id']}")
        print(f"Run ID: {context['run_id']}")
        if context["parent_run_id"]:
            print(f"Parent Run ID: {context['parent_run_id']}")
    
    print(f"\nCluster ID: {context['cluster_id']}")
    print(f"Workspace URL: {context['workspace_url']}")
    
    print("="*80)